# C08 · Tick wraparound, debouncing, memory pools and stack guards

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/01_c_bare_metal/C08_tick_wraparound_debouncing_memory_pools_and_stac.ipynb)

**Track:** C and bare-metal · **Skill axis (sheet):** C and bare-metal · **Time:** about 60 min

**Skill Hub concepts:** Wraparound-safe time comparison; Debouncing; Fixed-block memory pool; Stack overflow detection

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("C08")

## Map: four small patterns every firmware has

| Pattern | Rule | Classic bug |
|---|---|---|
| Tick comparison | `elapsed = now - start` in unsigned; `after(a, b) = (int32_t)(a - b) > 0` | `if (now > deadline)` breaks at wrap |
| Debounce | accept a new level only after N equal samples in a row | reacting to the first edge of a bounce |
| Fixed-block pool | free list threaded through the free blocks: O(1) alloc and free, no fragmentation | `malloc` in an interrupt or control loop |
| Stack guard | paint the stack with a pattern, measure how much was overwritten (high-water mark) | stack silently grows into `.bss` |

Wraparound: a 32-bit millisecond tick wraps after $2^{32}\ \text{ms} \approx 49.7$ days. Unsigned subtraction is
arithmetic modulo $2^{32}$, so `now - start` is right across the wrap as long as the true interval is below $2^{31}$.

```
Fixed-block pool, 4 blocks:      free -> [B0] -> [B2] -> [B3] -> NULL     (B1 is in use)
each free block's first word holds the pointer to the next free block
```

## Task 1: all four in one file

In [ ]:
%%writefile patterns.c
#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

/* ---- time ---- */
uint32_t elapsed(uint32_t now, uint32_t start) { (void)now; (void)start; return 0; /* TODO */ }
bool time_after(uint32_t a, uint32_t b) {      /* true if a is later than b */
    (void)a; (void)b; return false; /* TODO */
}

/* ---- debounce: output changes only after `need` consecutive samples that differ from it ---- */
typedef struct { bool out; uint8_t count; uint8_t need; } Debounce;
bool debounce_update(Debounce *d, bool raw) {
    (void)raw; return d->out; /* TODO */
}

/* ---- fixed-block pool ---- */
typedef struct { void *free; size_t block; } Pool;
void pool_init(Pool *p, void *buf, size_t block, size_t count) {
    (void)p; (void)buf; (void)block; (void)count; /* TODO: thread a free list through the blocks */
}
void *pool_alloc(Pool *p) {
    (void)p; return NULL; /* TODO */
}
void pool_free(Pool *p, void *blk) {
    (void)p; (void)blk; /* TODO */
}

/* ---- stack painting: stack[0] is the deepest word (stack grows down toward it) ---- */
#define PAINT 0xDEADBEEFu
void stack_paint(uint32_t *stack, size_t words) { for (size_t i = 0; i < words; i++) stack[i] = PAINT; }
size_t stack_high_water(const uint32_t *stack, size_t words) {   /* words ever used */
    (void)stack; (void)words; return 0; /* TODO */
}

In [ ]:
lab.check("patterns")

## Checkpoint: Math
**M1 (Apply).** Days until a 32-bit millisecond counter wraps (2 decimals)?

**M2 (Apply).** `time_after(0x00000010, 0xFFFFFFF0)` as 1 or 0?

**M3 (Analyze).** A debouncer samples every 5 ms and needs 4 consecutive differing samples. The switch settles right
after a sample is taken. How many milliseconds until the output changes, in the worst case?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Evaluate).** Why do hard real-time loops use a fixed-block pool instead of `malloc`?
- a) it uses less RAM in every case
- b) constant-time alloc and free with no fragmentation, so worst-case timing and memory are known
- c) `malloc` cannot return aligned memory

**K2 (Understand).** Compare three ways to detect stack overflow on a Cortex-M with FreeRTOS: painting with a
high-water mark, a canary checked at context switch, and an MPU guard region. What does each catch, and when?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `c08.h`).
3. In `roboprep/checks/c08.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()